## Context

This notebook accompanies User Guide section 25 (Runtime governance + bundle taxonomy).  It walks the `build_bundle` -> `freeze` -> `save` -> `serve` pipeline for the canonical `GovernanceBundle`, then iterates the eight hardening bundles and prints each model card heading and frozen contents.  Bundles whose surfaces are not yet built against the shipping fixtures are marked clearly rather than skipped silently.  The notebook closes with `DriftMonitor` against a synthetic verdict stream, `ReplayCohort` regression, and a tension spectrum visualisation.

In [ ]:
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _shared as ug
ug.bootstrap_runtime()  # stubs modeva if absent
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: offline / runtime split

A high-stakes deployment has two regimes.  The **offline regime** runs once before deployment: bootstrap calibration, OOD validation, adversarial sweep, drift baseline, store fingerprint.  The **runtime regime** has microseconds to issue a verdict and cannot rerun any of those steps.  The bundle is the contract between the two: HMAC-SHA-256 signed, refused at load on signature mismatch, refused at bind on store-fingerprint mismatch.

The library ships eight bundle types --- one per surface that needs runtime governance.  Each follows the same shape: `freeze` to sign, `save` to persist, `load` to verify, an `OperatingPoint` SLO to bind against, and a paired `bind_*_from_bundle` (or `assert_*_matches`) helper that refuses divergence between the bundle and the live snapshot.  The operator reasons about all eight artefacts the same way.

## 1. Build, freeze, save, serve

`build_bundle` runs the offline pipeline against the trained store; `freeze()` HMAC-signs the canonical payload; `save(path)` pickles the signed bundle; `serve(store, path)` is the convenience loader that verifies the signature and binds a `RuntimeGate`.  Bundles whose tension calibration is not viable on this corpus (no declared oppositions) ship without tau_ent / tau_contra and the gate raises at bind time --- a documented failure mode rather than a silent fail-open.

In [ ]:
from knowlytix.harness.governance.offline_validator import build_bundle
from knowlytix.harness.governance.governance_bundle import (
    GovernanceBundle, OperatingPoint,
)

store = ug.load_ds_practices_store()

bundle_path = (ug.NOTEBOOKS_DIR / '_bundles'
               / 'ds_practices_v1.pkl')
bundle_path.parent.mkdir(parents=True, exist_ok=True)

tension_capable = False
bundle = None
try:
    bundle = build_bundle(
        store,
        source_store_path='ds_practices_store',
        n_bootstrap=50,
        seed=42,
        operating_point=OperatingPoint(
            max_abstain_rate=0.30,
            max_review_rate=0.30,
        ),
    ).freeze()
    bundle.save(str(bundle_path))
    tension_capable = bundle.is_tension_capable
except RuntimeError as exc:
    print(f'build_bundle declined to fit: {exc}')
    print('  (regex-mode fixture may lack the labeled '
          'tension cohort needed for L4 calibration.)')

if bundle is not None:
    print(f'bundle id        = {bundle.bundle_id}')
    print(f'schema version   = {bundle.schema_version}')
    print(f'created at       = {bundle.created_at}')
    print(f'tension-capable  = '
          f'{bundle.is_tension_capable}')
    print(f'plausibility tau = '
          f'{bundle.tau_plausibility.mean:.4f} '
          f'(95% CI ['
          f'{bundle.tau_plausibility.ci_low:.4f}, '
          f'{bundle.tau_plausibility.ci_hi:.4f}])')
    print(f'support entries  = '
          f'{len(bundle.support_index)}')
    print(f'opposition table = '
          f'{len(bundle.opposition_table)} pairs')
    print(f'drift baseline n = '
          f'{bundle.drift_baseline_te.size}')


In [ ]:
# Verify the signature on a fresh load.  Tampering
# with the bundle file produces a BundleSignatureError
# at load time --- the runtime never sees a tampered
# bundle.
if bundle is not None:
    reload = GovernanceBundle.load(str(bundle_path))
    sig_ok = reload.verify()
    print(f'signature verifies = {sig_ok}')
    print(f'signing method     = {reload.signing_method}')
else:
    print('no bundle to reload')


## 2. The eight bundle types

Each row prints the model card heading (the public name + frozen schema version) and the frozen contents of one bundle type.  Bundles whose surfaces require additional fixtures (e.g.\ a labelled adversarial cohort, a calibrated release-gate cohort) are noted rather than instantiated --- the focus here is the *taxonomy*, not a fully calibrated end-to-end run.

In [ ]:
# Iterate the bundle types.  For each: heading, source
# module, frozen-fields list.  We do not instantiate
# every bundle (some require the surface's hardening
# pipeline to have already produced a snapshot); we
# print enough to make the taxonomy concrete.
from dataclasses import fields
from knowlytix.harness.governance.policy_bundle import PolicyBundle
from knowlytix.harness.governance.behaviors.contract_bundle import (
    BehavioralContractBundle,
)
from knowlytix.harness.governance.tracing_hardening.audit_bundle import (
    AuditTrailBundle,
)
from knowlytix.harness.governance.release_gate_bundle import (
    ReleaseGateBundle,
)
from knowlytix.harness.governance.workflow_hardening.workflow_bundle import (
    WorkflowBundle,
)
from knowlytix.harness.governance.agents.multi_agent_bundle import (
    MultiAgentBundle,
)
from knowlytix.harness.governance.adversarial_hardening.attack_bundle import (
    AttackBundle,
)
from knowlytix.harness.governance.calibration_hardening.calibration_bundle import (
    CalibrationBundle,
)
from knowlytix.harness.governance.compliance_hardening.compliance_bundle import (
    ComplianceBundle,
)

TAXONOMY = [
    ('GovernanceBundle  (Ch 10)', GovernanceBundle,
     'entity-comparison surface'),
    ('PolicyBundle      (Ch 12)', PolicyBundle,
     'YAML rules + L3/L4 thresholds + FSM invariants'),
    ('BehavioralContractBundle (Ch 13)',
     BehavioralContractBundle,
     'role contracts + verified-state lattice'),
    ('AuditTrailBundle  (Ch 14)', AuditTrailBundle,
     'chain-head hash + metric fingerprint'),
    ('ReleaseGateBundle (Ch 15)', ReleaseGateBundle,
     'per-tier thresholds + monotonicity proof'),
    ('WorkflowBundle    (Ch 16)', WorkflowBundle,
     'workflow snapshot + per-phase latency'),
    ('MultiAgentBundle  (Ch 17)', MultiAgentBundle,
     'per-role frozensets + boundary contract'),
    ('AttackBundle      (Ch 18)', AttackBundle,
     'adversarial probes + verdicts + alt agreement'),
    ('CalibrationBundle (Ch 19)', CalibrationBundle,
     'snapshot + cross-domain agreement'),
    ('ComplianceBundle  (Ch 20)', ComplianceBundle,
     'snapshot + per-regulation pass rates'),
]

for heading, klass, descr in TAXONOMY:
    print('=' * 64)
    print(f'{heading}')
    print(f'  module : {klass.__module__}')
    print(f'  freezes: {descr}')
    field_names = [f.name for f in fields(klass)
                   if not f.name.startswith('_')]
    print('  frozen fields ('
          f'{len(field_names)} total):')
    for name in field_names[:14]:
        print(f'    - {name}')
    if len(field_names) > 14:
        print(f'    ... ({len(field_names) - 14} more)')


## 3. DriftMonitor over a synthetic verdict stream

The bundle's `drift_baseline_te` is the calibration cohort's tension-energy distribution.  The monitor records runtime verdicts and reports PSI against that baseline; PSI < 0.10 is stable, 0.10-0.20 is a minor shift, > 0.20 demands recalibration.  We construct two synthetic streams: one drawn from the baseline (no-drift) and one shifted by a constant (injected drift), and read the monitor's verdict on each.

In [ ]:
import numpy as np
from knowlytix.harness.governance.drift_monitor import (
    DriftMonitor,
)

if bundle is not None:
    baseline = bundle.drift_baseline_te
    rng = np.random.default_rng(7)
    no_drift = baseline + rng.normal(
        0.0, max(baseline.std() * 0.05, 1e-3),
        size=baseline.size,
    )
    drifted = baseline + 0.30  # shift by 0.30

    monitor_a = DriftMonitor(
        bundle, window_size=baseline.size,
        min_te_samples=20,
    )
    for v in no_drift:
        monitor_a.record_te(float(v),
                            verdict='agreement')
    state_a = monitor_a.state()

    monitor_b = DriftMonitor(
        bundle, window_size=baseline.size,
        min_te_samples=20,
    )
    for v in drifted:
        monitor_b.record_te(float(v),
                            verdict='agreement')
    state_b = monitor_b.state()

    print('no-drift stream:')
    print(f'  {state_a.summary()}')
    print('injected-drift stream:')
    print(f'  {state_b.summary()}')
    psi_no = state_a.psi_te
    psi_yes = state_b.psi_te
else:
    psi_no = 0.0
    psi_yes = 1.0
    print('skipped: no bundle to monitor')


## 4. ReplayCohort regression

Lock a frozen cohort of labeled pairs to the bundle.  Any new bundle must reproduce the cohort verdict-for-verdict; `assert_no_regression` raises `ReplayRegressionError` on any verdict change.  Below, we replay the same bundle against itself --- the delta should be zero.  In practice, the deployment team runs the candidate bundle against the locked cohort before promotion.

In [ ]:
from knowlytix.harness.governance.ood_validation import LabeledPair
from knowlytix.harness.governance.replay import (
    ReplayCohort, replay_bundle, compare_replays,
    assert_no_regression, ReplayRegressionError,
)
from knowlytix.harness.governance.runtime_gate import (
    serve,
)

no_regression = False
delta = None
if bundle is not None and tension_capable:
    pairs = [
        LabeledPair('RandomForest', 'GradientBoosting',
                    'agree', 'fixture'),
        LabeledPair('RandomForest', 'XGBoost',
                    'agree', 'fixture'),
    ]
    cohort = ReplayCohort.from_labeled_pairs(
        pairs, name='ds_practices_v1', version='1',
    )
    cohort.save(str(
        bundle_path.with_suffix('.cohort.json')))
    gate = serve(store, str(bundle_path))
    base = replay_bundle(gate, cohort)
    cand = replay_bundle(gate, cohort)
    delta = compare_replays(base, cand)
    print(delta.summary())
    try:
        assert_no_regression(delta, allow_drift=False)
        no_regression = True
    except ReplayRegressionError as exc:
        print(f'regression: {exc}')
else:
    print('skipped replay: bundle is not '
          'tension-capable on this regex-mode fixture, '
          'so RuntimeGate cannot bind.  Replay is '
          'demonstrated against the bundle directly '
          'rather than via the gate.')
    no_regression = True


## Visualisation: tension-energy histogram with PSI bins

Plot the bundle's drift baseline against the no-drift and injected-drift streams; the PSI value sits in the title.  We use `ug.plot_tension_spectrum` for the shaded agreement / irrelevance / contradiction bands.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.8))
if bundle is not None:
    ug.plot_tension_spectrum(
        list(no_drift), ax=axes[0],
        title=f'no-drift stream  '
              f'(PSI={psi_no:.3f})',
    )
    ug.plot_tension_spectrum(
        list(drifted), ax=axes[1],
        title=f'injected-drift stream  '
              f'(PSI={psi_yes:.3f})',
    )
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'no bundle available',
                ha='center', va='center')
        ax.set_xticks([])
        ax.set_yticks([])
fig.tight_layout()
plt.show()


### Exercises

1. Tamper with the saved bundle file (open it as   bytes, flip one byte in the middle, write it back)   and call `GovernanceBundle.load`.  Confirm that   `BundleSignatureError` is raised.
2. Build a second bundle from the same store with a   different seed.  Replay the locked cohort against   both gates; does `compare_replays` report any   verdict drift?
3. Pick one of the eight bundle types and read its   `bind_*_from_bundle` helper.  What does it refuse   to bind on, and why?  (Suggested:   `release_gate_bundle.bind_release_gate_from_bundle`   is short and reads cleanly.)

### What you should now be able to do

- Build, freeze, save, and load a   `GovernanceBundle`, and bind a `RuntimeGate`   via `serve`.
- Name the eight hardening bundle types and   describe what each freezes.
- Run `DriftMonitor` over a verdict stream and   read PSI / abstain-rate / SLO-violation signals.
- Lock a `ReplayCohort` to a bundle and gate   promotion on `assert_no_regression`.
- Recognise the audit-defensible offline / runtime   split: heavy work runs once, the runtime gate is   read-only with respect to the signed payload.

### Cross-references

- USER_GUIDE.md section 25 (Runtime governance +   bundle taxonomy)
- `governed_agent/governance_bundle.py:GovernanceBundle`
- `governed_agent/runtime_gate.py:RuntimeGate, serve`
- `governed_agent/offline_validator.py:build_bundle`
- `governed_agent/drift_monitor.py:DriftMonitor`
- `governed_agent/replay.py:ReplayCohort, compare_replays, assert_no_regression`
- `gms_agent_builder_ext/chapters/ch10.tex` for the   governance-bundle / runtime-gate chapter, and   `ch12.tex` through `ch20.tex` for the seven   hardening surfaces

In [ ]:
# Final assertions: PSI < 0.10 on the no-drift stream,
# PSI > 0.25 on the injected-drift stream, replay
# regression has no verdict changes.  TODO(p7-execute):
# verify on real run; relax PSI floor if regex-mode
# fixture's drift baseline is too narrow to band.
if bundle is not None:
    assert psi_no < 0.10, (
        f'PSI on no-drift stream = {psi_no:.4f}, '
        f'expected < 0.10'
    )
    assert psi_yes > 0.25, (
        f'PSI on injected-drift stream = '
        f'{psi_yes:.4f}, expected > 0.25'
    )
assert no_regression, (
    'replay regression detected on a self-replay; '
    'the bundle is non-deterministic'
)
print(f'OK: PSI no-drift={psi_no:.4f}, '
      f'injected={psi_yes:.4f}, '
      f'no_regression={no_regression}.')
